# Clean segment-level model analysis with explicit whitelist variables

This notebook reads the segment-level modeling table and runs OLS plus optional XGBoost SHAP.

Variable selection is whitelist-based. The model uses only the variables listed in `X_GROUPS`; no dynamic regex discovery is used.

Default target is `speed_kmh`. Other possible targets are left as comments in the control panel.

Main outputs are saved under `outputs_new_clear/model_analysis_graphml_roadclass/`.


In [ ]:
# =========================================================
# 0. Imports and global controls
#    Python 3.8 compatible
# =========================================================
import json
import math
import os
import re
import warnings
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------
# Modeling target
# ---------------------------------------------------------
# Main target for this notebook.
Y_COL = "speed_kmh"

# Other possible targets. Keep them here for later robustness models.
# Y_COL = "overspeed_20"
# Y_COL = "duration"
# Y_COL = "final_distance_m"

Y_TRANSFORM = "raw"  # one of: raw, log, log1p, sqrt, signed_log1p

# Optional row filter. Example: "speed_kmh >= 3.6 and speed_kmh <= 60".
# Leave as None to use the full model table. The upstream filtered table usually already applies this filter.
ROW_QUERY = None

# Cluster-robust standard errors for OLS.
# If this column is absent or has too few groups, the notebook falls back to HC3 robust errors.
CLUSTER_COL = "courier_id"

# ---------------------------------------------------------
# Explicit whitelist X variables
# ---------------------------------------------------------
# The model uses only variables listed in X_GROUPS, plus optional X_EXTRA.
# No dynamic regex discovery is used.
#
# Important exclusions already applied here:
# speed_kmh, final_distance_m, duration, straight_distance_m,
# time_pressure_sec, is_workday_local, path_len_m_from_edges,
# n_conedges_in_segment, highly duplicated rider behavior variables,
# and highly collinear local road count variables.

GRAPHML_ROAD_CLASS_SHARE_COLS = [
    "road_class_share_levelFourRoad_lenw_mean",
    "road_class_share_levelThreeRoad_lenw_mean",
    "road_class_share_secondaryRoad_lenw_mean",
    "road_class_share_nationalRoad_lenw_mean",
    "road_class_share_provincialRoad_lenw_mean",
    "road_class_share_overPass_lenw_mean",
]

POI_LANDUSE_COLS = [
    # "poi_count_300m_lenw_mean",
    "poi_mix_entropy_norm_300m_lenw_mean",
    "poi_count_restaurant_300m_lenw_mean",
    # "restaurant_shp_count_300m_lenw_mean",
]

X_GROUPS = OrderedDict([
    ("trip_state", [
        "duration",
        "onhand_order_count_start",
        "time_pressure_min",
        "is_weekend_local",
        # "final_distance_m",
        # "start_hour_sin",
        # "start_hour_cos",
    ]),
    ("rider_behavior", [
        "rider_avg_orders_per_active_day",
        # "rider_active_day_share_in_data",
        # "rider_median_onhand_raw",
        # "rider_median_max_onhand_per_wave",
        # "rider_share_batch_wave_ge2",
        # "rider_median_segment_distance_m",
    ]),
    # ("road_class_shares", GRAPHML_ROAD_CLASS_SHARE_COLS),
    ("road_topology_geometry", [
        # "degree_mean_end_lenw_mean",
        # "betweenness_raw_mean_end_lenw_mean",
        # "closeness_approx_mean_end_lenw_mean",
        # "tortuosity_lenw_mean",
        "curvature_deg_per_m_lenw_mean",
        # "road_len_in_300m_lenw_mean",
        "intersection_density_per_km_300m_lenw_mean",
    ]),
    ("landuse", POI_LANDUSE_COLS),
])

# Optional manual additions. Use this only for robustness models.
X_EXTRA = [
    # "straight_distance_m",
]

# Optional manual removals from the whitelist above.
X_DROP = []

# Columns treated as categorical controls. Binary 0/1 variables can stay numeric unless you prefer dummies.
CATEGORICAL_VARS = []

# Road-class shares sum to one when all listed classes are present.
# With an intercept, one road-class column should be omitted as the reference category.
DROP_ONE_ROAD_CLASS_REFERENCE = True
ROAD_CLASS_REFERENCE_PREFERENCE = [
    "road_class_share_levelFourRoad_lenw_mean",
]
ROAD_CLASS_REFERENCE_MODE = "preference_then_largest"  # one of: preference_then_largest, auto_largest, auto_smallest

# Columns that must never enter X even if accidentally added through X_EXTRA.
PROTECTED_NON_X = [
    Y_COL,
    "overspeed_20",
    "speed_kmh",
    # "duration",
    "final_distance_m",
    "path_len_m_from_edges",
    "dist_network_only",
    "start_time",
    "end_time",
    "parent_final_distance_m",
    "parent_duration",
]

# ---------------------------------------------------------
# Transform controls
# ---------------------------------------------------------
# Supported rules: raw, log, log1p, sqrt, signed_log1p, logit01, standardize.
# For OLS, transformed columns replace the raw columns in X. The raw columns remain in the data table.
VARIABLE_TRANSFORMS = OrderedDict([
    # ("rider_avg_orders_per_active_day", "log1p"),
    # ("rider_total_orders_raw", "log1p"),
    # ("rider_mean_segment_distance_m", "log1p"),
    # ("betweenness_raw_mean_end_lenw_mean", "log1p"),
    # ("road_len_in_300m_lenw_mean", "log1p"),
    # ("intersection_count_in_300m_lenw_mean", "log1p"),
    # ("connectivity_sum_degree_in_300m_lenw_mean", "log1p"),
    # ("poi_count_300m_lenw_mean", "log1p"),
    # ("restaurant_shp_count_300m_lenw_mean", "log1p"),
])

# No pattern transforms. Add transforms explicitly below when needed.
PATTERN_VARIABLE_TRANSFORMS = OrderedDict([])

# The audit table will recommend additional transformations from distribution shape.
# Keep this False for reproducibility, then copy useful recommendations into VARIABLE_TRANSFORMS.
AUTO_APPLY_RECOMMENDED_TRANSFORMS = False
RECOMMEND_SKEW_THRESHOLD = 1.0
RECOMMEND_MIN_UNIQUE = 20

# VIF handling.
AUTO_DROP_HIGH_VIF = False
VIF_THRESHOLD = 10.0
VIF_PROTECTED = ["const"]

# OLS options.
RUN_OLS = True
OLS_STANDARDIZE_X_FOR_ESTIMATION = False
SAVE_DIAGNOSTIC_PLOTS = True
PLOT_SAMPLE_N = 80000

# XGBoost and SHAP options.
RUN_XGBOOST_SHAP = True
XGB_TEST_SIZE = 0.20
XGB_MAX_ROWS = 250000
SHAP_SAMPLE_N = 20000
SHAP_TOP_N = 20
XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 4,
    "learning_rate": 0.04,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 1.0,
    "min_child_weight": 5,
    "objective": "reg:squarederror",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}


In [52]:
# =========================================================
# 1. Paths
# =========================================================
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR.parent,
    Path.cwd(),
    Path("/mnt/data"),
]

OUTPUT_ROOT = BASE_DIR / "outputs_new_clear" / "model_analysis_graphml_roadclass"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TRIP_TABLE_CANDIDATES = [
    "outputs_new_clear/segment_variables_graphml_roadclass/segments/segment_model_table.csv.gz",
    "outputs_new_clear/segment_variables_graphml_roadclass/segments/segment_model_table.csv",
    "outputs_new_clear/segment_variables_clean/segments/segment_model_table.csv.gz",
    "outputs_new_clear/segment_variables_clean/segments/segment_model_table.csv",
    "outputs/segment_variables_clean/segments/segment_model_table.csv.gz",
    "outputs/segment_variables_clean/segments/segment_model_table.csv",
    "segment_model_table.csv.gz",
    "segment_model_table.csv",
]


def resolve_path(candidates, search_dirs=None, required=True):
    if search_dirs is None:
        search_dirs = SEARCH_DIRS
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]
    checked = []
    for cand in candidates:
        p = Path(cand)
        if p.is_absolute():
            checked.append(str(p))
            if p.exists():
                return p
    for root in search_dirs:
        root = Path(root)
        for cand in candidates:
            p = root / cand
            checked.append(str(p))
            if p.exists():
                return p
    if required:
        raise FileNotFoundError("Could not find any candidate path. Checked:\n" + "\n".join(checked))
    return None

TRIP_TABLE_PATH = resolve_path(TRIP_TABLE_CANDIDATES, required=True)

PATH_INFO = {
    "trip_table_path": str(TRIP_TABLE_PATH),
    "output_root": str(OUTPUT_ROOT),
}
(OUTPUT_ROOT / "effective_paths.json").write_text(
    json.dumps(PATH_INFO, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Trip table:", TRIP_TABLE_PATH)
print("Output root:", OUTPUT_ROOT)


Trip table: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\segments\segment_model_table.csv.gz
Output root: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\model_analysis_graphml_roadclass


In [53]:
# =========================================================
# 2. General helper functions
# =========================================================
def ensure_dir(path_like):
    Path(path_like).mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def save_text(text, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")


def read_table(path):
    path = Path(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)


def write_csv_atomic(df, path, **kwargs):
    path = Path(path)
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, **kwargs)
    os.replace(str(tmp), str(path))


def make_unique_column_names(cols):
    seen = {}
    out = []
    for c in list(cols):
        c = str(c)
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append("%s__dup%d" % (c, seen[c]))
    return out


def pick_existing(df, cols):
    return [c for c in cols if c in df.columns]


def discover_group_vars(group_name, existing_cols):
    existing_cols = list(existing_cols)
    out = []
    for pat in DYNAMIC_X_GROUP_PATTERNS.get(group_name, []):
        rx = re.compile(pat)
        for c in existing_cols:
            if rx.search(str(c)):
                out.append(c)
    return list(OrderedDict.fromkeys(out))


def is_road_class_share_col(c):
    return re.match(r"^road_class_share_.+_lenw_mean$", str(c)) is not None


def choose_road_class_reference(df, road_cols):
    road_cols = [c for c in road_cols if c in df.columns]
    if len(road_cols) <= 1:
        return None
    if ROAD_CLASS_REFERENCE_MODE == "preference_then_largest":
        for c in ROAD_CLASS_REFERENCE_PREFERENCE:
            if c in road_cols:
                return c
    means = df[road_cols].apply(pd.to_numeric, errors="coerce").mean().sort_values(ascending=False)
    if ROAD_CLASS_REFERENCE_MODE == "auto_smallest":
        means = means.sort_values(ascending=True)
    if len(means) == 0:
        return road_cols[-1]
    return str(means.index[0])


def transform_rule_for_variable(var, manual_rules):
    if var in manual_rules:
        return manual_rules.get(var, "raw")
    for pat, rule in PATTERN_VARIABLE_TRANSFORMS.items():
        if re.search(pat, str(var)):
            return rule
    return "raw"


def drop_constant_columns(df):
    keep = []
    dropped = []
    for c in df.columns:
        nun = df[c].nunique(dropna=True)
        if nun <= 1:
            dropped.append(c)
        else:
            keep.append(c)
    return df[keep].copy(), dropped


def build_selected_vars(group_dict, groups_to_use, extra_vars, drop_vars, existing_cols):
    cols = []
    for g in groups_to_use:
        cols.extend(group_dict.get(g, []))
        cols.extend(discover_group_vars(g, existing_cols))
    cols.extend(extra_vars or [])
    existing = set(existing_cols)
    drop_set = set(drop_vars or [])
    protected = set(PROTECTED_NON_X or [])
    out = []
    for c in cols:
        if c in existing and c not in drop_set and c not in protected:
            out.append(c)
    return list(OrderedDict.fromkeys(out))


def numeric_summary_table(df, cols):
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        non = s.dropna()
        if len(non) == 0:
            rows.append({
                "variable": c,
                "n": 0,
                "n_unique": 0,
                "min": np.nan,
                "p01": np.nan,
                "p50": np.nan,
                "p99": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "std": np.nan,
                "skew": np.nan,
                "share_zero": np.nan,
            })
            continue
        rows.append({
            "variable": c,
            "n": int(non.shape[0]),
            "n_unique": int(non.nunique(dropna=True)),
            "min": float(non.min()),
            "p01": float(non.quantile(0.01)),
            "p50": float(non.quantile(0.50)),
            "p99": float(non.quantile(0.99)),
            "max": float(non.max()),
            "mean": float(non.mean()),
            "std": float(non.std(ddof=0)),
            "skew": float(non.skew()) if non.nunique() > 2 else np.nan,
            "share_zero": float(np.mean(non == 0)),
        })
    return pd.DataFrame(rows)


def recommend_transform_for_series(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "raw", "all_missing"
    nun = int(s.nunique(dropna=True))
    mn = float(s.min())
    mx = float(s.max())
    if nun <= 2:
        return "raw", "binary_or_constant"
    if nun < RECOMMEND_MIN_UNIQUE:
        return "raw", "low_cardinality"
    skew = float(s.skew()) if nun > 2 else 0.0
    if mn >= 0 and abs(skew) >= RECOMMEND_SKEW_THRESHOLD:
        if mn == 0:
            return "log1p", "nonnegative_with_zero_and_high_skew"
        return "log", "positive_and_high_skew"
    if mn < 0 and abs(skew) >= RECOMMEND_SKEW_THRESHOLD:
        return "signed_log1p", "has_negative_values_and_high_skew"
    return "raw", "no_strong_transform_signal"


def build_transform_audit(df, variables, manual_rules):
    rows = []
    for c in variables:
        if c not in df.columns:
            continue
        rec, reason = recommend_transform_for_series(df[c])
        manual = transform_rule_for_variable(c, manual_rules)
        final = manual
        if AUTO_APPLY_RECOMMENDED_TRANSFORMS and manual == "raw":
            final = rec
        rows.append({
            "variable": c,
            "manual_rule": manual,
            "recommended_rule": rec,
            "recommend_reason": reason,
            "final_rule_if_applied": final,
        })
    return pd.DataFrame(rows)


In [54]:
# =========================================================
# 3. Transformation and design-matrix helpers
# =========================================================
def transform_series(s, rule):
    s = pd.to_numeric(s, errors="coerce")
    rule = str(rule or "raw").lower()
    if rule == "raw":
        return s, "raw"
    if rule == "log":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s > 0
        out.loc[mask] = np.log(s.loc[mask])
        return out, "log_positive_only"
    if rule == "log1p":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s >= 0
        out.loc[mask] = np.log1p(s.loc[mask])
        return out, "log1p_nonnegative_only"
    if rule == "sqrt":
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = s >= 0
        out.loc[mask] = np.sqrt(s.loc[mask])
        return out, "sqrt_nonnegative_only"
    if rule == "signed_log1p":
        return np.sign(s) * np.log1p(np.abs(s)), "signed_log1p"
    if rule == "logit01":
        eps = 1e-6
        out = pd.Series(np.nan, index=s.index, dtype=float)
        mask = (s >= 0) & (s <= 1)
        v = s.loc[mask].clip(eps, 1.0 - eps)
        out.loc[mask] = np.log(v / (1.0 - v))
        return out, "logit01_clipped"
    if rule == "standardize":
        mu = s.mean()
        sd = s.std(ddof=0)
        if pd.isna(sd) or sd <= 0:
            return s * np.nan, "standardize_failed"
        return (s - mu) / sd, "standardize"
    raise ValueError("Unknown transform rule: %s" % rule)


def transformed_name(var, rule):
    rule = str(rule or "raw").lower()
    if rule == "raw":
        return var
    prefixes = {
        "log": "ln_",
        "log1p": "ln1p_",
        "sqrt": "sqrt_",
        "signed_log1p": "slog1p_",
        "logit01": "logit_",
        "standardize": "z_",
    }
    return prefixes.get(rule, rule + "_") + var


def apply_transform_rules(df, variables, manual_rules, transform_audit=None, y_col=None, y_rule="raw"):
    out = df.copy()
    rows = []
    final_rules = {}

    audit_map = {}
    if transform_audit is not None and len(transform_audit) > 0:
        for r in transform_audit.itertuples(index=False):
            audit_map[getattr(r, "variable")] = getattr(r, "final_rule_if_applied")

    for c in variables:
        if c not in out.columns:
            continue
        rule = transform_rule_for_variable(c, manual_rules)
        if AUTO_APPLY_RECOMMENDED_TRANSFORMS and c in audit_map and rule == "raw":
            rule = audit_map[c]
        final_rules[c] = rule
        new_c = transformed_name(c, rule)
        if rule == "raw":
            rows.append({
                "variable": c,
                "new_variable": c,
                "rule": "raw",
                "status": "kept_raw",
                "n_missing_after": int(out[c].isna().sum()),
            })
            continue
        try:
            transformed, status = transform_series(out[c], rule)
            out[new_c] = transformed
            rows.append({
                "variable": c,
                "new_variable": new_c,
                "rule": rule,
                "status": status,
                "n_missing_after": int(pd.isna(transformed).sum()),
            })
        except Exception as exc:
            rows.append({
                "variable": c,
                "new_variable": c,
                "rule": rule,
                "status": "failed_%s" % repr(exc),
                "n_missing_after": int(out[c].isna().sum()),
            })
            final_rules[c] = "raw"
    return out, final_rules, pd.DataFrame(rows)


def replace_with_transformed_names(cols, rules):
    out = []
    for c in cols:
        rule = rules.get(c, "raw")
        out.append(transformed_name(c, rule))
    return out


def build_design_matrix(df, cont_cols, cat_cols=None, add_const=True):
    cat_cols = cat_cols or []
    parts = []
    if len(cont_cols) > 0:
        cont_df = df[cont_cols].apply(pd.to_numeric, errors="coerce")
        if cont_df.columns.duplicated().any():
            cont_df.columns = make_unique_column_names(cont_df.columns)
        parts.append(cont_df)
    for c in cat_cols:
        if c not in df.columns:
            continue
        tmp = df[c]
        if pd.api.types.is_numeric_dtype(tmp):
            tmp = tmp.astype("Int64").astype(str)
        else:
            tmp = tmp.astype(str)
        dummies = pd.get_dummies(tmp, prefix=c, drop_first=True)
        if dummies.columns.duplicated().any():
            dummies.columns = make_unique_column_names(dummies.columns)
        parts.append(dummies)
    if len(parts) == 0:
        X = pd.DataFrame(index=df.index)
    else:
        X = pd.concat(parts, axis=1)
    X = X.apply(pd.to_numeric, errors="coerce")
    X, dropped_const = drop_constant_columns(X)
    if add_const:
        X = sm.add_constant(X, has_constant="add")
    return X, dropped_const


def drop_exact_alias_columns(X, protect_cols=None, tol=1e-12):
    protect_cols = protect_cols or []
    ordered_cols = [c for c in X.columns if c in protect_cols] + [c for c in X.columns if c not in protect_cols]
    X = X[ordered_cols].copy()
    if X.shape[1] <= 1:
        return X, []
    vals = X.to_numpy(dtype=float)
    keep_idx = []
    dropped = []
    current_rank = 0
    for j, c in enumerate(X.columns):
        sub_idx = keep_idx + [j]
        sub_vals = vals[:, sub_idx]
        rank = int(np.linalg.matrix_rank(sub_vals, tol=tol))
        if rank > current_rank:
            keep_idx.append(j)
            current_rank = rank
        else:
            dropped.append(c)
    keep_cols = list(X.columns[keep_idx])
    return X[keep_cols].copy(), dropped


def compute_vif(X):
    X_vif = X.copy()
    for c in ["const", "intercept"]:
        if c in X_vif.columns:
            X_vif = X_vif.drop(columns=[c])
    X_vif = X_vif.apply(pd.to_numeric, errors="coerce")
    X_vif = X_vif.replace([np.inf, -np.inf], np.nan)
    X_vif = X_vif.fillna(X_vif.mean())
    X_vif, _ = drop_constant_columns(X_vif)
    if X_vif.shape[1] == 0:
        return pd.DataFrame(columns=["variable", "VIF"])
    vals = X_vif.to_numpy(dtype=float)
    rows = []
    for i, col in enumerate(X_vif.columns):
        try:
            v = float(variance_inflation_factor(vals, i))
        except Exception:
            v = np.inf
        rows.append({"variable": col, "VIF": v})
    return pd.DataFrame(rows).sort_values(["VIF", "variable"], ascending=[False, True]).reset_index(drop=True)


def iterative_drop_high_vif(X, threshold=10.0, protected=None):
    protected = set(protected or [])
    X_cur = X.copy()
    dropped = []
    while True:
        vif = compute_vif(X_cur)
        if vif.empty:
            break
        vif = vif[~vif["variable"].isin(protected)].copy()
        if vif.empty:
            break
        top = vif.iloc[0]
        if not np.isfinite(top["VIF"]) or top["VIF"] > threshold:
            col = top["variable"]
            if col in X_cur.columns:
                X_cur = X_cur.drop(columns=[col])
                dropped.append({"variable": col, "VIF_at_drop": top["VIF"]})
                continue
        break
    return X_cur, pd.DataFrame(dropped)


def standardize_columns(X, cols):
    X = X.copy()
    rows = []
    for c in cols:
        if c not in X.columns or c == "const":
            continue
        s = pd.to_numeric(X[c], errors="coerce")
        mu = float(s.mean())
        sd = float(s.std(ddof=0))
        if not np.isfinite(sd) or sd <= 0:
            continue
        X[c] = (s - mu) / sd
        rows.append({"column": c, "mean": mu, "std": sd})
    return X, pd.DataFrame(rows)


In [55]:
# =========================================================
# 4. Load data and build the modeling table
# =========================================================
def load_and_filter_trip_table():
    df = read_table(TRIP_TABLE_PATH)
    print("Raw trip table shape:", df.shape)

    if ROW_QUERY is not None:
        before = len(df)
        df = df.query(ROW_QUERY).copy()
        print("After ROW_QUERY: %s -> %s" % (before, len(df)))

    if Y_COL not in df.columns:
        raise KeyError("Y_COL not found in input table: %s" % Y_COL)

    return df


def flatten_whitelist_groups(groups, existing_cols):
    existing_cols = set(existing_cols)
    rows = []
    selected = []
    missing = []
    for group_name, cols in groups.items():
        for c in cols:
            if c in existing_cols:
                if c not in selected:
                    selected.append(c)
                rows.append({"group_name": group_name, "variable": c, "exists": 1})
            else:
                missing.append({"group_name": group_name, "variable": c, "exists": 0})
                rows.append({"group_name": group_name, "variable": c, "exists": 0})
    return selected, pd.DataFrame(rows), pd.DataFrame(missing)


def prepare_modeling_data(df):
    x_raw, selected_groups_all, missing_whitelist = flatten_whitelist_groups(X_GROUPS, df.columns)

    # Optional manual additions and removals.
    for c in X_EXTRA:
        if c in df.columns and c not in x_raw:
            x_raw.append(c)
            selected_groups_all = pd.concat([
                selected_groups_all,
                pd.DataFrame([{"group_name": "x_extra", "variable": c, "exists": 1}])
            ], ignore_index=True)

    protected = set(PROTECTED_NON_X)
    x_drop_set = set(X_DROP) | protected
    x_raw = [c for c in x_raw if c not in x_drop_set]

    # Road class reference category.
    road_class_cols_selected = [c for c in x_raw if is_road_class_share_col(c)]
    road_class_reference = None
    if DROP_ONE_ROAD_CLASS_REFERENCE and len(road_class_cols_selected) > 1:
        road_class_reference = choose_road_class_reference(df, road_class_cols_selected)
        if road_class_reference in x_raw:
            x_raw = [c for c in x_raw if c != road_class_reference]
            print("Dropped road-class reference column:", road_class_reference)

    cat_cols = pick_existing(df, CATEGORICAL_VARS)

    selected_groups = selected_groups_all[selected_groups_all["exists"] == 1].copy()
    selected_groups["included_in_x_raw"] = selected_groups["variable"].isin(x_raw).astype(int)
    dynamic_discovered = pd.DataFrame(columns=["group_name", "variable"])

    road_class_diagnostics = pd.DataFrame({
        "road_class_column": road_class_cols_selected,
        "included_in_x_after_reference_drop": [c in x_raw for c in road_class_cols_selected],
        "is_reference": [c == road_class_reference for c in road_class_cols_selected],
        "mean_share": [pd.to_numeric(df[c], errors="coerce").mean() if c in df.columns else np.nan for c in road_class_cols_selected],
    })

    # Audit distributions before changing anything.
    audit_vars = list(OrderedDict.fromkeys(x_raw + [Y_COL]))
    summary = numeric_summary_table(df, audit_vars)
    transform_audit = build_transform_audit(df, x_raw, VARIABLE_TRANSFORMS)

    # Apply X transforms.
    df2, x_rules, x_transform_info = apply_transform_rules(
        df,
        variables=x_raw,
        manual_rules=VARIABLE_TRANSFORMS,
        transform_audit=transform_audit,
    )
    x_model = replace_with_transformed_names(x_raw, x_rules)
    x_model = [c for c in x_model if c in df2.columns]

    # Apply Y transform separately.
    y_new = transformed_name(Y_COL, Y_TRANSFORM)
    if Y_TRANSFORM == "raw":
        df2[y_new] = pd.to_numeric(df2[Y_COL], errors="coerce")
        y_info = pd.DataFrame([{
            "variable": Y_COL,
            "new_variable": y_new,
            "rule": "raw",
            "status": "kept_raw",
            "n_missing_after": int(df2[y_new].isna().sum()),
        }])
    else:
        y_transformed, y_status = transform_series(df2[Y_COL], Y_TRANSFORM)
        df2[y_new] = y_transformed
        y_info = pd.DataFrame([{
            "variable": Y_COL,
            "new_variable": y_new,
            "rule": Y_TRANSFORM,
            "status": y_status,
            "n_missing_after": int(df2[y_new].isna().sum()),
        }])

    needed = [y_new] + x_model + cat_cols
    if CLUSTER_COL in df2.columns:
        needed.append(CLUSTER_COL)
    before = len(df2)
    df_model = df2.dropna(subset=[c for c in needed if c in df2.columns]).copy()
    print("Rows after dropping model missing values: %s -> %s" % (before, len(df_model)))
    print("Selected raw X count:", len(x_raw))
    print("Selected model X count:", len(x_model))
    if len(missing_whitelist) > 0:
        print("Missing whitelist variables:", len(missing_whitelist))
        display(missing_whitelist)

    return {
        "df_model": df_model,
        "x_raw": x_raw,
        "x_model": x_model,
        "cat_cols": cat_cols,
        "y_model": y_new,
        "summary": summary,
        "transform_audit": transform_audit,
        "x_transform_info": x_transform_info,
        "y_transform_info": y_info,
        "selected_groups": selected_groups,
        "missing_whitelist": missing_whitelist,
        "dynamic_discovered": dynamic_discovered,
        "road_class_diagnostics": road_class_diagnostics,
        "road_class_reference": road_class_reference,
    }

trip_df = load_and_filter_trip_table()
prepared = prepare_modeling_data(trip_df)

ensure_dir(OUTPUT_ROOT / "diagnostics")
write_csv_atomic(prepared["summary"], OUTPUT_ROOT / "diagnostics" / "selected_variable_raw_summary.csv", index=False)
write_csv_atomic(prepared["transform_audit"], OUTPUT_ROOT / "diagnostics" / "transform_recommendations.csv", index=False)
write_csv_atomic(prepared["x_transform_info"], OUTPUT_ROOT / "diagnostics" / "x_transform_info.csv", index=False)
write_csv_atomic(prepared["y_transform_info"], OUTPUT_ROOT / "diagnostics" / "y_transform_info.csv", index=False)
write_csv_atomic(prepared["selected_groups"], OUTPUT_ROOT / "diagnostics" / "selected_variables_by_group.csv", index=False)
write_csv_atomic(prepared["missing_whitelist"], OUTPUT_ROOT / "diagnostics" / "missing_whitelist_variables.csv", index=False)
write_csv_atomic(prepared["dynamic_discovered"], OUTPUT_ROOT / "diagnostics" / "dynamic_variables_discovered.csv", index=False)
write_csv_atomic(prepared["road_class_diagnostics"], OUTPUT_ROOT / "diagnostics" / "road_class_model_diagnostics.csv", index=False)

print("Road-class reference column:", prepared["road_class_reference"])
print("Selected raw X count:", len(prepared["x_raw"]))
prepared["transform_audit"].head(30)


Raw trip table shape: (453375, 118)
Rows after dropping model missing values: 453375 -> 453375
Selected raw X count: 8
Selected model X count: 8
Road-class reference column: None
Selected raw X count: 8


,variable,manual_rule,recommended_rule,recommend_reason,final_rule_if_applied
0,onhand_order_count_start,raw,raw,low_cardinality,raw
1,time_pressure_min,raw,raw,no_strong_transform_signal,raw
2,is_weekend_local,raw,raw,binary_or_constant,raw
3,rider_avg_orders_per_active_day,raw,raw,no_strong_transform_signal,raw
4,curvature_deg_per_m_lenw_mean,raw,log1p,nonnegative_with_zero_and_high_skew,raw
5,intersection_density_per_km_300m_lenw_mean,raw,raw,no_strong_transform_signal,raw
6,poi_mix_entropy_norm_300m_lenw_mean,raw,signed_log1p,has_negative_values_and_high_skew,raw
7,poi_count_restaurant_300m_lenw_mean,raw,raw,binary_or_constant,raw


## Transformation notes

The table above flags variables that look like good transformation candidates. A practical rule is: count variables, distance-like variables, and highly skewed nonnegative centrality measures often work better with `log1p`; binary variables and proportions are usually easier to interpret raw.

To change the actual model, edit `VARIABLE_TRANSFORMS`. Keep `AUTO_APPLY_RECOMMENDED_TRANSFORMS = False` unless you want the notebook to use the automatic recommendations directly.


In [56]:
# =========================================================
# 5. OLS functions
# =========================================================
def matrix_condition_number(X):
    try:
        return float(np.linalg.cond(X.to_numpy(dtype=float)))
    except Exception:
        return np.nan


def coef_table_from_ols(res):
    params = pd.Series(res.params)
    se = pd.Series(res.bse, index=params.index)
    tvals = pd.Series(res.tvalues, index=params.index)
    pvals = pd.Series(res.pvalues, index=params.index)
    ci = res.conf_int()

    if isinstance(ci, np.ndarray):
        ci = pd.DataFrame(ci, index=params.index, columns=["ci_low", "ci_high"])
    else:
        ci.columns = ["ci_low", "ci_high"]

    return pd.DataFrame({
        "variable": params.index,
        "coef": params.values,
        "std_err": se.values,
        "t_value": tvals.values,
        "p_value": pvals.values,
        "ci_low": ci["ci_low"].values,
        "ci_high": ci["ci_high"].values,
    })


def get_x_group_lookup():
    """
    Build variable to group lookup from X_GROUPS.

    This lets the coefficient table follow the same conceptual order as the model variable whitelist.
    """
    lookup = {}
    order_lookup = {}

    if "X_GROUPS" not in globals():
        return lookup, order_lookup

    group_order = 1
    for group_name, cols in X_GROUPS.items():
        for within_order, col in enumerate(cols):
            lookup[col] = group_name
            order_lookup[col] = (group_order, within_order)
        group_order += 1

    return lookup, order_lookup


def infer_base_variable_from_design_col(var_name, x_group_lookup):
    """
    Map transformed design-matrix columns back to their original variable names.

    Exact continuous variables match directly.
    Dummy variables are matched by prefix when categorical columns are used.
    """
    if var_name == "const":
        return "const"

    if var_name in x_group_lookup:
        return var_name

    for base_var in x_group_lookup.keys():
        prefixes = [
            base_var + "_",
            base_var + "[",
            "C(" + base_var + ")",
            "C(" + base_var + ")[",
        ]
        for prefix in prefixes:
            if str(var_name).startswith(prefix):
                return base_var

    return var_name


def add_coef_group_order(coef):
    """
    Add group labels and sorting keys to the coefficient table.
    """
    coef = coef.copy()
    x_group_lookup, x_order_lookup = get_x_group_lookup()

    group_names = []
    base_vars = []
    group_orders = []
    within_orders = []

    for var in coef["variable"].astype(str).values:
        if var == "const":
            base_var = "const"
            group_name = "constant"
            group_order = 0
            within_order = 0
        else:
            base_var = infer_base_variable_from_design_col(var, x_group_lookup)
            group_name = x_group_lookup.get(base_var, "other")
            group_order, within_order = x_order_lookup.get(base_var, (999, 999))

        base_vars.append(base_var)
        group_names.append(group_name)
        group_orders.append(group_order)
        within_orders.append(within_order)

    coef["variable_base"] = base_vars
    coef["variable_group"] = group_names
    coef["_group_order"] = group_orders
    coef["_within_group_order"] = within_orders

    coef = coef.sort_values(
        ["_group_order", "_within_group_order", "variable"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    return coef


def round_numeric_columns(df, digits=3):
    """
    Round real numeric columns for readable output tables.

    Boolean columns are intentionally skipped because pandas may classify bool as numeric,
    but Series.round can fail on bool dtype in some environments.
    """
    out = df.copy()

    for col in out.columns:
        s = out[col]

        if pd.api.types.is_bool_dtype(s):
            continue

        if pd.api.types.is_numeric_dtype(s):
            out[col] = pd.to_numeric(s, errors="coerce").round(digits)

    return out


def format_p_values_for_display(df, p_col="p_value"):
    """
    Keep p values numeric and rounded for CSV output.
    Very small p values will appear as 0.000 after rounding.
    """
    out = df.copy()

    if p_col in out.columns:
        out[p_col] = pd.to_numeric(out[p_col], errors="coerce").round(3)

    return out


def reorder_coef_columns(coef):
    front_cols = [
        "variable_group",
        "variable",
        "variable_base",
        "coef",
        "std_err",
        "t_value",
        "p_value",
        "ci_low",
        "ci_high",
        "y_col",
        "x_standardized_for_estimation",
    ]
    front_cols = [c for c in front_cols if c in coef.columns]
    rest_cols = [c for c in coef.columns if c not in front_cols and not c.startswith("_")]
    return coef[front_cols + rest_cols].copy()


def plot_ols_diagnostics(res, outdir, stem):
    ensure_dir(outdir)

    resid = pd.Series(np.asarray(res.resid)).astype(float)
    fitted = pd.Series(np.asarray(res.fittedvalues)).astype(float)

    if len(resid) > PLOT_SAMPLE_N:
        idx = np.random.RandomState(RANDOM_SEED).choice(
            len(resid),
            size=PLOT_SAMPLE_N,
            replace=False
        )
        resid_plot = resid.iloc[idx]
        fitted_plot = fitted.iloc[idx]
    else:
        resid_plot = resid
        fitted_plot = fitted

    resid_std = (resid_plot - resid_plot.mean()) / (resid_plot.std(ddof=0) + 1e-12)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(fitted_plot, resid_std, s=5, alpha=0.25)
    ax.axhline(0)
    ax.set_xlabel("fitted")
    ax.set_ylabel("standardized residual")
    ax.set_title("Residuals vs fitted")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_vs_fitted.png"), dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(resid_std, bins=60)
    ax.set_xlabel("standardized residual")
    ax.set_ylabel("count")
    ax.set_title("Residual histogram")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_hist.png"), dpi=220)
    plt.close(fig)

    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111)

    if len(resid_std) > 50000:
        qq = resid_std.sample(50000, random_state=RANDOM_SEED)
    else:
        qq = resid_std

    st.probplot(qq, dist="norm", plot=ax)
    ax.set_title("Residual QQ plot")
    fig.tight_layout()
    fig.savefig(Path(outdir) / (stem + "__resid_qq.png"), dpi=220)
    plt.close(fig)

    return pd.DataFrame({
        "metric": ["n", "resid_mean", "resid_std", "resid_min", "resid_max"],
        "value": [len(resid), resid.mean(), resid.std(ddof=0), resid.min(), resid.max()],
    })


def run_ols(prep):
    df = prep["df_model"].copy()
    y_col = prep["y_model"]
    x_cols = prep["x_model"]
    cat_cols = prep["cat_cols"]

    y = pd.to_numeric(df[y_col], errors="coerce")

    X, dropped_const = build_design_matrix(
        df,
        cont_cols=x_cols,
        cat_cols=cat_cols,
        add_const=True
    )
    X, dropped_alias = drop_exact_alias_columns(X, protect_cols=["const"])

    # Align y and X after all transformations.
    use_idx = y.dropna().index.intersection(X.dropna().index)
    y = y.loc[use_idx]
    X = X.loc[use_idx]
    df_fit = df.loc[use_idx].copy()

    vif_before = compute_vif(X)

    dropped_vif = pd.DataFrame(columns=["variable", "VIF_at_drop"])
    if AUTO_DROP_HIGH_VIF:
        X, dropped_vif = iterative_drop_high_vif(
            X,
            threshold=VIF_THRESHOLD,
            protected=VIF_PROTECTED
        )

    vif_after = compute_vif(X)

    if OLS_STANDARDIZE_X_FOR_ESTIMATION:
        scale_cols = [c for c in X.columns if c != "const"]
        X_fit, scale_df = standardize_columns(X, scale_cols)
    else:
        X_fit = X.copy()
        scale_df = pd.DataFrame(columns=["column", "mean", "std"])

    cov_type = "HC3"
    cov_kwds = None

    if CLUSTER_COL in df_fit.columns:
        groups = df_fit[CLUSTER_COL]
        if groups.nunique(dropna=True) > 1:
            cov_type = "cluster"
            cov_kwds = {"groups": groups}

    model = sm.OLS(y.astype(float), X_fit.astype(float))

    if cov_kwds is None:
        res = model.fit(cov_type=cov_type)
    else:
        res = model.fit(cov_type=cov_type, cov_kwds=cov_kwds)

    coef = coef_table_from_ols(res)
    coef["y_col"] = y_col

    if len(scale_df):
        coef["x_standardized_for_estimation"] = coef["variable"].isin(set(scale_df["column"]))
    else:
        coef["x_standardized_for_estimation"] = False

    coef = add_coef_group_order(coef)
    coef = reorder_coef_columns(coef)

    fit_info = pd.DataFrame([{
        "model_name": "ols_main",
        "y_col": y_col,
        "y_base": Y_COL,
        "y_transform": Y_TRANSFORM,
        "n_obs": int(len(y)),
        "n_params": int(X_fit.shape[1]),
        "r2": float(res.rsquared),
        "adj_r2": float(res.rsquared_adj),
        "aic": float(res.aic),
        "bic": float(res.bic),
        "cov_type": cov_type,
        "cluster_col": CLUSTER_COL if cov_type == "cluster" else None,
        "condition_number": matrix_condition_number(X_fit),
        "dropped_constant_cols": json.dumps(dropped_const, ensure_ascii=False),
        "dropped_alias_cols": json.dumps(dropped_alias, ensure_ascii=False),
        "auto_dropped_high_vif_cols": json.dumps(dropped_vif.to_dict(orient="records"), ensure_ascii=False),
    }])

    outdir = OUTPUT_ROOT / "ols"
    ensure_dir(outdir)

    # Full precision outputs.
    write_csv_atomic(coef, outdir / "ols_coef_table_full_precision.csv", index=False)
    write_csv_atomic(fit_info, outdir / "ols_fit_info_full_precision.csv", index=False)
    write_csv_atomic(vif_before, outdir / "vif_before_optional_drop_full_precision.csv", index=False)
    write_csv_atomic(vif_after, outdir / "vif_final_full_precision.csv", index=False)

    # Readable rounded outputs.
    coef_rounded = round_numeric_columns(coef, digits=3)
    coef_rounded = format_p_values_for_display(coef_rounded, p_col="p_value")

    fit_info_rounded = round_numeric_columns(fit_info, digits=3)
    vif_before_rounded = round_numeric_columns(vif_before, digits=3)
    vif_after_rounded = round_numeric_columns(vif_after, digits=3)
    dropped_vif_rounded = round_numeric_columns(dropped_vif, digits=3)

    write_csv_atomic(coef_rounded, outdir / "ols_coef_table.csv", index=False)
    write_csv_atomic(fit_info_rounded, outdir / "ols_fit_info.csv", index=False)
    write_csv_atomic(vif_before_rounded, outdir / "vif_before_optional_drop.csv", index=False)
    write_csv_atomic(vif_after_rounded, outdir / "vif_final.csv", index=False)
    write_csv_atomic(dropped_vif_rounded, outdir / "dropped_high_vif.csv", index=False)

    if len(scale_df):
        scale_df_rounded = round_numeric_columns(scale_df, digits=3)
        write_csv_atomic(scale_df, outdir / "x_standardization_info_full_precision.csv", index=False)
        write_csv_atomic(scale_df_rounded, outdir / "x_standardization_info.csv", index=False)

    try:
        save_text(str(res.summary()), outdir / "ols_summary.txt")
    except Exception:
        pass

    if SAVE_DIAGNOSTIC_PLOTS:
        diag = plot_ols_diagnostics(res, outdir, "ols_main")
        diag_rounded = round_numeric_columns(diag, digits=3)
        write_csv_atomic(diag, outdir / "ols_diagnostics_summary_full_precision.csv", index=False)
        write_csv_atomic(diag_rounded, outdir / "ols_diagnostics_summary.csv", index=False)

    return {
        "result": res,
        "coef": coef,
        "coef_rounded": coef_rounded,
        "fit_info": fit_info,
        "fit_info_rounded": fit_info_rounded,
        "vif_before": vif_before,
        "vif_after": vif_after,
        "X": X,
        "X_fit": X_fit,
        "y": y,
        "df_fit": df_fit,
    }


if RUN_OLS:
    ols_results = run_ols(prepared)
    display(ols_results["fit_info_rounded"])
    display(ols_results["coef_rounded"].head(40))
else:
    ols_results = None
    print("RUN_OLS is False. Skipped OLS.")

,model_name,y_col,y_base,y_transform,n_obs,n_params,r2,adj_r2,aic,bic,cov_type,cluster_col,condition_number,dropped_constant_cols,dropped_alias_cols,auto_dropped_high_vif_cols
0,ols_main,speed_kmh,speed_kmh,raw,453375,8,0.079,0.079,2670664.433,2670752.629,cluster,courier_id,328.67,"[""poi_count_restaurant_300m_lenw_mean""]",[],[]


,variable_group,variable,variable_base,coef,std_err,t_value,p_value,ci_low,ci_high,y_col,x_standardized_for_estimation
0,constant,const,const,20.709,0.128,161.249,0.000,20.457,20.960,speed_kmh,False
1,trip_state,onhand_order_count_start,onhand_order_count_start,0.119,0.011,10.967,0.000,0.097,0.140,speed_kmh,False
2,trip_state,time_pressure_min,time_pressure_min,-0.008,0.001,-7.361,0.000,-0.010,-0.006,speed_kmh,False
3,trip_state,is_weekend_local,is_weekend_local,0.010,0.019,0.542,0.588,-0.027,0.048,speed_kmh,False
4,rider_behavior,rider_avg_orders_per_active_day,rider_avg_orders_per_active_day,-0.009,0.002,-4.503,0.000,-0.013,-0.005,speed_kmh,False
5,road_topology_geometry,curvature_deg_per_m_lenw_mean,curvature_deg_per_m_lenw_mean,-8.274,0.076,-109.441,0.000,-8.422,-8.126,speed_kmh,False
6,road_topology_geometry,intersection_density_per_km_300m_lenw_mean,intersection_density_per_km_300m_lenw_mean,-0.722,0.011,-66.945,0.000,-0.743,-0.700,speed_kmh,False
7,landuse,poi_mix_entropy_norm_300m_lenw_mean,poi_mix_entropy_norm_300m_lenw_mean,-0.665,0.088,-7.524,0.000,-0.838,-0.492,speed_kmh,False


## XGBoost and SHAP

This section uses the same selected variables and transformations as the OLS section. XGBoost is useful here because it can capture nonlinear effects and interactions. SHAP values then help diagnose which variables drive predictions and where nonlinear thresholds appear.

For large data, `XGB_MAX_ROWS` and `SHAP_SAMPLE_N` keep runtime manageable. Increase them when you need a more stable SHAP profile.


In [57]:
# =========================================================
# 6. XGBoost and SHAP functions
# =========================================================
def import_ml_dependencies():
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        import xgboost as xgb
        import shap
        return {
            "ok": True,
            "train_test_split": train_test_split,
            "mean_absolute_error": mean_absolute_error,
            "mean_squared_error": mean_squared_error,
            "r2_score": r2_score,
            "xgb": xgb,
            "shap": shap,
            "error": None,
        }
    except Exception as exc:
        return {"ok": False, "error": repr(exc)}


def build_ml_matrix(prep):
    df = prep["df_model"].copy()
    y_col = prep["y_model"]
    x_cols = prep["x_model"]
    cat_cols = prep["cat_cols"]

    y = pd.to_numeric(df[y_col], errors="coerce")
    X, _ = build_design_matrix(df, cont_cols=x_cols, cat_cols=cat_cols, add_const=False)
    X = X.replace([np.inf, -np.inf], np.nan)
    use_idx = y.dropna().index.intersection(X.dropna().index)
    X = X.loc[use_idx].astype(float)
    y = y.loc[use_idx].astype(float)
    df_fit = df.loc[use_idx].copy()

    if len(X) > XGB_MAX_ROWS:
        sample_idx = X.sample(XGB_MAX_ROWS, random_state=RANDOM_SEED).index
        X = X.loc[sample_idx]
        y = y.loc[sample_idx]
        df_fit = df_fit.loc[sample_idx]
        print("XGBoost sampled rows:", len(X))

    return X, y, df_fit


def run_xgboost_shap(prep):
    deps = import_ml_dependencies()
    outdir = OUTPUT_ROOT / "xgboost_shap"
    ensure_dir(outdir)

    if not deps["ok"]:
        msg = "Skipped XGBoost SHAP because dependencies are unavailable: %s" % deps["error"]
        print(msg)
        save_text(msg, outdir / "skipped.txt")
        return None

    train_test_split = deps["train_test_split"]
    mean_absolute_error = deps["mean_absolute_error"]
    mean_squared_error = deps["mean_squared_error"]
    r2_score = deps["r2_score"]
    xgb = deps["xgb"]
    shap = deps["shap"]

    X, y, df_fit = build_ml_matrix(prep)
    if X.shape[0] < 100 or X.shape[1] == 0:
        msg = "Skipped XGBoost SHAP because model matrix is too small. Shape: %s" % (str(X.shape),)
        print(msg)
        save_text(msg, outdir / "skipped.txt")
        return None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=XGB_TEST_SIZE,
        random_state=RANDOM_SEED,
    )

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train)

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    metrics = pd.DataFrame([
        {
            "split": "train",
            "n": int(len(y_train)),
            "rmse": float(math.sqrt(mean_squared_error(y_train, pred_train))),
            "mae": float(mean_absolute_error(y_train, pred_train)),
            "r2": float(r2_score(y_train, pred_train)),
        },
        {
            "split": "test",
            "n": int(len(y_test)),
            "rmse": float(math.sqrt(mean_squared_error(y_test, pred_test))),
            "mae": float(mean_absolute_error(y_test, pred_test)),
            "r2": float(r2_score(y_test, pred_test)),
        },
    ])
    write_csv_atomic(metrics, outdir / "xgb_metrics.csv", index=False)

    importance = pd.DataFrame({
        "feature": X.columns,
        "importance_gain_like": model.feature_importances_,
    }).sort_values("importance_gain_like", ascending=False)
    write_csv_atomic(importance, outdir / "xgb_feature_importance.csv", index=False)

    # SHAP sample.
    if len(X_test) > SHAP_SAMPLE_N:
        X_shap = X_test.sample(SHAP_SAMPLE_N, random_state=RANDOM_SEED)
    else:
        X_shap = X_test.copy()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_arr = np.asarray(shap_values)

    shap_imp = pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_arr).mean(axis=0),
        "mean_shap": shap_arr.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    write_csv_atomic(shap_imp, outdir / "shap_importance.csv", index=False)

    # Summary plots.
    try:
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_arr, X_shap, show=False, max_display=SHAP_TOP_N)
        plt.tight_layout()
        plt.savefig(outdir / "shap_summary_beeswarm.png", dpi=220, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        save_text(repr(exc), outdir / "shap_summary_beeswarm_failed.txt")

    try:
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_arr, X_shap, show=False, plot_type="bar", max_display=SHAP_TOP_N)
        plt.tight_layout()
        plt.savefig(outdir / "shap_summary_bar.png", dpi=220, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        save_text(repr(exc), outdir / "shap_summary_bar_failed.txt")

    # Dependence plots for top features.
    top_features = list(shap_imp["feature"].head(SHAP_TOP_N))
    dep_dir = outdir / "dependence_plots"
    ensure_dir(dep_dir)
    for feat in top_features:
        try:
            plt.figure(figsize=(7, 5))
            shap.dependence_plot(feat, shap_arr, X_shap, show=False, interaction_index=None)
            plt.tight_layout()
            safe = re.sub(r"[^A-Za-z0-9_]+", "_", str(feat))[:120]
            plt.savefig(dep_dir / (safe + "__dependence.png"), dpi=220, bbox_inches="tight")
            plt.close()
        except Exception as exc:
            safe = re.sub(r"[^A-Za-z0-9_]+", "_", str(feat))[:120]
            save_text(repr(exc), dep_dir / (safe + "__failed.txt"))

    # Save predictions for test set.
    pred_df = pd.DataFrame({
        "row_index": X_test.index,
        "y_true": y_test.values,
        "y_pred": pred_test,
    })
    write_csv_atomic(pred_df, outdir / "xgb_test_predictions.csv", index=False)

    return {
        "model": model,
        "metrics": metrics,
        "importance": importance,
        "shap_importance": shap_imp,
        "X_test": X_test,
        "y_test": y_test,
    }

if RUN_XGBOOST_SHAP:
    xgb_results = run_xgboost_shap(prepared)
    if xgb_results is not None:
        display(xgb_results["metrics"])
        display(xgb_results["shap_importance"].head(20))
else:
    xgb_results = None
    print("RUN_XGBOOST_SHAP is False. Skipped XGBoost SHAP.")


XGBoost sampled rows: 250000


,split,n,rmse,mae,r2
0,train,200000,4.317473,3.417938,0.185807
1,test,50000,4.370320,3.447513,0.175702


,feature,mean_abs_shap,mean_shap
4,curvature_deg_per_m_lenw_mean,0.943487,0.020442
5,intersection_density_per_km_300m_lenw_mean,0.600137,0.007950
6,poi_mix_entropy_norm_300m_lenw_mean,0.449550,-0.003671
1,time_pressure_min,0.281259,-0.008527
3,rider_avg_orders_per_active_day,0.196086,-0.000566
0,onhand_order_count_start,0.114342,0.000786
2,is_weekend_local,0.002781,0.000007


<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

In [58]:
# =========================================================
# 7. Save effective config and final pointers
# =========================================================
effective_config = {
    "trip_table_path": str(TRIP_TABLE_PATH),
    "output_root": str(OUTPUT_ROOT),
    "y_col": Y_COL,
    "y_transform": Y_TRANSFORM,
    "row_query": ROW_QUERY,
    "cluster_col": CLUSTER_COL,
    "x_groups": {k: list(v) for k, v in X_GROUPS.items()},
    "x_whitelist_count_before_reference_drop": sum(len(v) for v in X_GROUPS.values()),
    "drop_one_road_class_reference": DROP_ONE_ROAD_CLASS_REFERENCE,
    "road_class_reference_mode": ROAD_CLASS_REFERENCE_MODE,
    "road_class_reference_used": prepared.get("road_class_reference") if "prepared" in globals() else None,
    "x_extra": X_EXTRA,
    "x_drop": X_DROP,
    "categorical_vars": CATEGORICAL_VARS,
    "variable_transforms": dict(VARIABLE_TRANSFORMS),
    "pattern_variable_transforms": dict(PATTERN_VARIABLE_TRANSFORMS),
    "auto_apply_recommended_transforms": AUTO_APPLY_RECOMMENDED_TRANSFORMS,
    "auto_drop_high_vif": AUTO_DROP_HIGH_VIF,
    "vif_threshold": VIF_THRESHOLD,
    "run_ols": RUN_OLS,
    "run_xgboost_shap": RUN_XGBOOST_SHAP,
    "xgb_params": XGB_PARAMS,
}
save_json(effective_config, OUTPUT_ROOT / "effective_config.json")

print("Done.")
print("OLS outputs:", OUTPUT_ROOT / "ols")
print("XGBoost and SHAP outputs:", OUTPUT_ROOT / "xgboost_shap")
print("Diagnostics:", OUTPUT_ROOT / "diagnostics")


Done.
OLS outputs: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\model_analysis_graphml_roadclass\ols
XGBoost and SHAP outputs: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\model_analysis_graphml_roadclass\xgboost_shap
Diagnostics: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\model_analysis_graphml_roadclass\diagnostics


## Explicit variable availability check

This block shows which explicitly listed variables exist in the model table and which are missing. Use it before running models when you change the variable notebook.


In [59]:
# =========================================================
# Explicit variable availability check
# =========================================================
def availability_table(df, groups):
    rows = []
    for group_name, cols in groups.items():
        for c in cols:
            rows.append({
                "group_name": group_name,
                "variable": c,
                "exists": int(c in df.columns),
                "non_null_share": float(df[c].notna().mean()) if c in df.columns else np.nan,
                "n_unique": int(df[c].nunique(dropna=True)) if c in df.columns else 0,
            })
    return pd.DataFrame(rows)

if "df" in globals():
    _model_df_for_check = df.copy()
elif "TRIP_TABLE_PATH" in globals():
    _model_df_for_check = read_table(TRIP_TABLE_PATH)
else:
    raise RuntimeError("Run the data loading cells first, or define TRIP_TABLE_PATH.")

var_availability = availability_table(_model_df_for_check, X_GROUPS)
write_csv_atomic(var_availability, OUTPUT_ROOT / "diagnostics" / "explicit_variable_availability.csv", index=False)
print("Missing explicit variables:")
display(var_availability[var_availability["exists"] == 0])
print("Available explicit variables:")
display(var_availability[var_availability["exists"] == 1].head(200))


Missing explicit variables:


,group_name,variable,exists,non_null_share,n_unique


Available explicit variables:


,group_name,variable,exists,non_null_share,n_unique
0,trip_state,duration,1,1.0,2488
1,trip_state,onhand_order_count_start,1,1.0,13
2,trip_state,time_pressure_min,1,1.0,6455
3,trip_state,is_weekend_local,1,1.0,2
4,rider_behavior,rider_avg_orders_per_active_day,1,1.0,842
5,road_topology_geometry,curvature_deg_per_m_lenw_mean,1,1.0,361014
6,road_topology_geometry,intersection_density_per_km_300m_lenw_mean,1,1.0,369522
7,landuse,poi_mix_entropy_norm_300m_lenw_mean,1,1.0,368432
8,landuse,poi_count_restaurant_300m_lenw_mean,1,1.0,1


## Final selected X and Y block

This block prints the exact whitelist configuration used by the notebook. Copy it back into the control panel when you want to freeze a model specification.


In [60]:
# =========================================================
# Final selected X and Y block for copy-paste
# =========================================================
def format_python_list(name, values, indent="    "):
    lines = [name + " = ["]
    for v in values:
        lines.append(indent + repr(str(v)) + ",")
    lines.append("]")
    return "\n".join(lines)

print("# Target")
print("Y_COL = %r" % Y_COL)
print("# Other possible targets:")
print("# Y_COL = 'overspeed_20'")
print("# Y_COL = 'duration'")
print("# Y_COL = 'final_distance_m'")
print()
print("# Raw X used after reference-category and protected-variable drops")
print(format_python_list("X_SELECTED_RAW", prepared["x_raw"]))
print()
print("# Model X after transforms")
print(format_python_list("X_SELECTED_MODEL", prepared["x_model"]))

copy_text = []
copy_text.append("Y_COL = %r" % Y_COL)
copy_text.append("")
copy_text.append(format_python_list("X_SELECTED_RAW", prepared["x_raw"]))
copy_text.append("")
copy_text.append(format_python_list("X_SELECTED_MODEL", prepared["x_model"]))
save_text("\n".join(copy_text), OUTPUT_ROOT / "diagnostics" / "final_selected_x_y_block.py")
print("Saved:", OUTPUT_ROOT / "diagnostics" / "final_selected_x_y_block.py")


# Target
Y_COL = 'speed_kmh'
# Other possible targets:
# Y_COL = 'overspeed_20'
# Y_COL = 'duration'
# Y_COL = 'final_distance_m'

# Raw X used after reference-category and protected-variable drops
X_SELECTED_RAW = [
    'onhand_order_count_start',
    'time_pressure_min',
    'is_weekend_local',
    'rider_avg_orders_per_active_day',
    'curvature_deg_per_m_lenw_mean',
    'intersection_density_per_km_300m_lenw_mean',
    'poi_mix_entropy_norm_300m_lenw_mean',
    'poi_count_restaurant_300m_lenw_mean',
]

# Model X after transforms
X_SELECTED_MODEL = [
    'onhand_order_count_start',
    'time_pressure_min',
    'is_weekend_local',
    'rider_avg_orders_per_active_day',
    'curvature_deg_per_m_lenw_mean',
    'intersection_density_per_km_300m_lenw_mean',
    'poi_mix_entropy_norm_300m_lenw_mean',
    'poi_count_restaurant_300m_lenw_mean',
]
Saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\model_analysis_graphml_roadclass\diagnostics\final_selected_x_y_block.py
